# 🚀 Kaggle 版 SDXL LoRA 训练 - 车柱完角色

Kaggle 每周 30 小时免费 T4 GPU，适合训练 LoRA。

**使用前：**
1. 右上角 Settings → Accelerator: GPU T4 x1
2. Persistence: 勾选（保存环境）
3. 按顺序运行所有 Cell

---

**数据上传方式：**
- 方法 A：Kaggle Data → Upload Dataset（推荐，稳定）
- 方法 B：直接拖拽文件到右侧文件面板


In [ ]:
# Step 1: 检查 GPU
import torch
import os

print("=" * 60)
print("📊 GPU 配置检查")
print("=" * 60)

print(f"✅ CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"✅ GPU: {gpu_name}")
    print(f"   显存: {gpu_memory:.2f} GB")
    
    # 测试 GPU 计算
    print("\n🧪 测试 GPU 计算...")
    x = torch.randn(1000, 1000).cuda()
    y = torch.matmul(x, x)
    print("   ✅ GPU 计算测试通过！")
else:
    print("\n❌ 错误: 未检测到 GPU！")
    print("   请点击右上角 Settings → Accelerator: GPU T4 x1")
    raise RuntimeError("No GPU detected")

print("\n" + "=" * 60)
print("✅ GPU 检查完成！")
print("=" * 60)


In [ ]:
# Step 2: 安装 kohya_ss 训练工具
import os
from pathlib import Path

print("=" * 60)
print("📦 安装 kohya_ss 训练工具")
print("=" * 60)

# 克隆 kohya_ss 仓库（含 sd-scripts 子模块）
if not Path("/kaggle/working/kohya_ss").exists():
    print("\n📥 克隆 kohya_ss 仓库（含 sd-scripts 子模块）...")
    !cd /kaggle/working && git clone --recurse-submodules https://github.com/bmaltais/kohya_ss.git
    print("   ✅ 克隆完成！")
else:
    print("\n✅ kohya_ss 已存在，更新...")
    %cd /kaggle/working/kohya_ss
    !git pull
    !git submodule update --init --recursive
    %cd /kaggle/working
    print("   ✅ 更新完成！")

# 安装依赖
print("\n📦 安装 Python 依赖...")
%cd /kaggle/working/kohya_ss
!pip install -q -r requirements.txt
!pip install -q -r requirements_linux.txt
print("   ✅ 依赖安装完成！")

# 安装 sd-scripts 训练脚本
print("\n📦 安装 sd-scripts 训练脚本...")
!pip install -q -e /kaggle/working/kohya_ss/sd-scripts
print("   ✅ sd-scripts 安装完成！")

# 安装 xformers（加速训练）
print("\n⚡ 安装 xformers（训练加速）...")
!pip install -q xformers
print("   ✅ xformers 安装完成！")

# 检查安装
print("\n🔍 检查安装...")
train_script_path = Path("/kaggle/working/kohya_ss/sd-scripts/train_network.py")
if train_script_path.exists():
    print(f"   ✅ train_network.py 存在: {train_script_path}")
else:
    print("   ❌ train_network.py 不存在！")
    raise FileNotFoundError("train_network.py not found")

%cd /kaggle/working

print("\n" + "=" * 60)
print("✅ kohya_ss 安装完成！")
print("=" * 60)


In [ ]:
# Step 3: 下载 SDXL 1.0 模型
from pathlib import Path

print("=" * 60)
print("📥 下载 SDXL 1.0 模型")
print("=" * 60)

# 创建模型目录
model_dir = Path("/kaggle/working/kohya_ss/models")
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "sd_xl_base_1.0.safetensors"

if model_path.exists():
    file_size = model_path.stat().st_size / 1024**3
    print(f"\n✅ 模型已存在: {file_size:.2f} GB")
else:
    print("\n📥 下载 SDXL 1.0 模型（~6.5GB，需要 5-10 分钟）...")
    !cd /kaggle/working/kohya_ss && wget -q --show-progress -O models/sd_xl_base_1.0.safetensors \
        https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors
    
    if model_path.exists():
        file_size = model_path.stat().st_size / 1024**3
        print(f"   ✅ 下载完成: {file_size:.2f} GB")
    else:
        print("   ❌ 下载失败！")
        raise RuntimeError("Model download failed")

print("\n" + "=" * 60)
print("✅ SDXL 模型准备完成！")
print("=" * 60)


In [ ]:
# Step 4: 上传训练数据
from pathlib import Path
import shutil
import os

print("=" * 60)
print("📁 准备训练数据")
print("=" * 60)

# 创建训练数据目录（100 = repeat count）
train_data_dir = Path("/kaggle/working/kohya_ss/train_data/100_chezhuwan")
train_data_dir.mkdir(parents=True, exist_ok=True)

print(f"\n✅ 训练数据目录: {train_data_dir}")
print("\n📌 上传方式（选一种）：")
print("   方法 A：右侧文件面板 → 上传 train_data/100_chezhuwan/")
print("   方法 B：Kaggle Data → 上传 Dataset → 符号链接到工作目录")
print("\n⚠️ 需要上传的文件：")
print("   - 图片文件：*.jpg, *.png（至少 20 张）")
print("   - 标签文件：*.txt（每张图片对应一个 .txt）")

# 检查数据是否已上传
image_files = list(train_data_dir.glob("*.jpg")) + list(train_data_dir.glob("*.png"))
txt_files = list(train_data_dir.glob("*.txt"))

print(f"\n📊 当前数据状态：")
print(f"   图片文件: {len(image_files)} 个")
print(f"   标签文件: {len(txt_files)} 个")

if len(image_files) == 0:
    print("\n⚠️ 警告: 没有找到图片文件！请先上传训练数据")
    print("   上传后，重新运行此 Cell")
elif len(image_files) != len(txt_files):
    print(f"\n⚠️ 警告: 图片和标签数量不匹配！")
else:
    print(f"\n✅ 数据完整！")
    if image_files:
        print(f"   示例: {image_files[0].name}")

print("\n" + "=" * 60)


In [ ]:
# Step 5: 创建训练配置文件
from pathlib import Path

print("=" * 60)
print("⚙️ 创建训练配置文件")
print("=" * 60)

config_content = """
[model]
pretrained_model_name_or_path = "/kaggle/working/kohya_ss/models/sd_xl_base_1.0.safetensors"
train_data_dir = "/kaggle/working/kohya_ss/train_data"
output_dir = "/kaggle/working/kohya_ss/output"
logging_dir = "/kaggle/working/kohya_ss/logs"

[training]
train_batch_size = 2
gradient_accumulation_steps = 1
network_dim = 32
network_alpha = 16
max_train_epochs = 20
learning_rate = 1e-4
mixed_precision = "fp16"
save_every_n_epochs = 5
save_last_n_epochs = 3
seed = 42

[advanced]
resolution = "512,512"
min_bucket_reso = 256
max_bucket_reso = 1024
bucket_reso_steps = 64
flip_aug = true
caption_dropout_rate = 0.1
cache_latents = true
cache_text_encoder_outputs = true
use_8bit_adam = true
xformers = true
lowram = false
"""

# 保存配置文件
config_dir = Path("/kaggle/working/kohya_ss/config")
config_dir.mkdir(parents=True, exist_ok=True)
config_file = config_dir / "chezhuwan_sdxl.toml"

with open(config_file, "w") as f:
    f.write(config_content)

print(f"\n✅ 配置文件已创建: {config_file}")
print(f"\n配置内容:")
print("-" * 60)
print(config_content)
print("-" * 60)

print("\n" + "=" * 60)
print("✅ 配置完成！")
print("=" * 60)


In [ ]:
# Step 6: 测试训练环境
from pathlib import Path
import subprocess
import torch

print("=" * 60)
print("🧪 测试训练环境")
print("=" * 60)

# 检查文件
print("\n🔍 检查文件...")
model_path = Path("/kaggle/working/kohya_ss/models/sd_xl_base_1.0.safetensors")
train_script = Path("/kaggle/working/kohya_ss/sd-scripts/train_network.py")
config_file = Path("/kaggle/working/kohya_ss/config/chezhuwan_sdxl.toml")
train_data_dir = Path("/kaggle/working/kohya_ss/train_data/100_chezhuwan")

checks = [
    (model_path, "SDXL 模型"),
    (train_script, "训练脚本"),
    (config_file, "配置文件"),
]

all_passed = True
for path, name in checks:
    if path.exists():
        print(f"   ✅ {name}: {path}")
    else:
        print(f"   ❌ {name}: 不存在！")
        all_passed = False

# 检查训练数据
print(f"\n📊 训练数据:")
if train_data_dir.exists():
    image_files = list(train_data_dir.glob("*.jpg")) + list(train_data_dir.glob("*.png"))
    txt_files = list(train_data_dir.glob("*.txt"))
    print(f"   图片: {len(image_files)} 个")
    print(f"   标签: {len(txt_files)} 个")
    
    if len(image_files) > 0:
        print(f"   示例图片: {image_files[0].name}")
        if txt_files:
            print(f"   示例标签: {txt_files[0].name}")
else:
    print(f"   ⚠️ 训练数据目录不存在: {train_data_dir}")
    all_passed = False

# 检查 GPU 显存
print(f"\n🎮 GPU 显存:")
if torch.cuda.is_available():
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"   总显存: {gpu_memory:.2f} GB")
    
    if gpu_memory < 8:
        print(f"   ⚠️ 显存不足 8GB，建议修改配置:")
        print(f"      train_batch_size = 1")
        print(f"      gradient_accumulation_steps = 2")
    else:
        print(f"   ✅ 显存充足（SDXL LoRA 需要 8GB+）")

print("\n" + "=" * 60)
if all_passed:
    print("✅ 环境测试通过！可以开始训练！")
else:
    print("❌ 环境测试失败！请检查上面的错误。")
print("=" * 60)


In [ ]:
# Step 7: 开始训练（完整训练）
from pathlib import Path
import subprocess
import os
import sys

print("=" * 60)
print("🚀 开始 SDXL LoRA 训练")
print("=" * 60)

# 检查环境
config_file = Path("/kaggle/working/kohya_ss/config/chezhuwan_sdxl.toml")
train_script = Path("/kaggle/working/kohya_ss/sd-scripts/train_network.py")

if not config_file.exists():
    print(f"❌ 配置文件不存在: {config_file}")
    raise FileNotFoundError("Config file not found")

if not train_script.exists():
    print(f"❌ 训练脚本不存在: {train_script}")
    raise FileNotFoundError("Train script not found")

# 切换到 kohya_ss 目录
os.chdir("/kaggle/working/kohya_ss")
print(f"\n📂 当前目录: {os.getcwd()}")

# 构建训练命令
cmd = [
    "accelerate", "launch",
    "--num_cpu_threads_per_process", "2",
    "sd-scripts/train_network.py",
    "--config_file", str(config_file)
]

print(f"\n🚀 开始训练...")
print(f"   命令: {' '.join(cmd)}")
print(f"\n⏳ 训练时间预估: 20 epochs ~ 1-2 小时")
print(f"   每 5 个 epoch 会自动保存一次模型\n")
print("-" * 60)

# 启动训练
try:
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        universal_newlines=True,
        bufsize=1
    )
    
    # 实时输出训练日志
    for line in process.stdout:
        print(line, end="")
    
    process.wait()
    
    if process.returncode == 0:
        print("\n" + "=" * 60)
        print("✅ 训练完成！")
        print("=" * 60)
    else:
        print(f"\n❌ 训练失败！返回码: {process.returncode}")
        
except Exception as e:
    print(f"\n❌ 训练出错: {e}")
    raise

print("\n📌 下一步：下载训练好的 LoRA 模型")
print("   运行下一个单元格下载模型")


In [ ]:
# Step 8: 下载训练好的 LoRA 模型
from pathlib import Path
from IPython.display import FileLink

print("=" * 60)
print("📥 下载 LoRA 模型")
print("=" * 60)
    
# 查找训练好的模型
output_dir = Path("/kaggle/working/kohya_ss/output")
lora_files = list(output_dir.glob("*.safetensors"))

if not lora_files:
    print("\n⚠️ 没有找到训练好的 LoRA 模型！")
    print(f"   请检查输出目录: {output_dir}")
else:
    print(f"\n✅ 找到 {len(lora_files)} 个 LoRA 模型:\n")
    
    for i, lora_file in enumerate(lora_files):
        file_size = lora_file.stat().st_size / 1024**2
        print(f"   [{i+1}] {lora_file.name} ({file_size:.2f} MB)")
    
    print(f"\n📥 生成下载链接...\n")
    
    # 生成下载链接
    for lora_file in lora_files:
        print(f"下载: {lora_file.name}")
        display(FileLink(str(lora_file)))
        print("")

print("\n" + "=" * 60)
print("✅ 完成！")
print("=" * 60)
print("\n💡 使用 LoRA 模型：")
print("   1. 在 Stable Diffusion WebUI 中加载 LoRA")
print("   2. 触发词: 车柱完")
   3. 调整 LoRA 权重（建议 0.6-0.8）")
